In [1]:
# for the extraction of curvature and head oscillation frequency
import math
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import os
from glob import glob
import matplotlib.pyplot as plt

C:\Users\Jalaja Madhusudhanan\AppData\Local\Temp\ipykernel_17588\2375325926.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [8]:
spline_files = [os.path.abspath(f) for f in glob(r"Y:\jalaja\behaviour\Autoscope\data\*w*\*Ch0\output\skeleton_spline_K_signed.csv")]
spline_files

['Y:\\jalaja\\behaviour\\Autoscope\\data\\27_11_22_w1\\2022-11-27_12-31_w1_Ch0\\output\\skeleton_spline_K_signed.csv',
 'Y:\\jalaja\\behaviour\\Autoscope\\data\\27_11_22_w2\\2022-11-27_13-19_w2_Ch0\\output\\skeleton_spline_K_signed.csv',
 'Y:\\jalaja\\behaviour\\Autoscope\\data\\27_11_22_w3\\2022-11-27_14-13_w3_Ch0\\output\\skeleton_spline_K_signed.csv',
 'Y:\\jalaja\\behaviour\\Autoscope\\data\\27_11_22_w4\\2022-11-27_15-07_w4_Ch0\\output\\skeleton_spline_K_signed.csv',
 'Y:\\jalaja\\behaviour\\Autoscope\\data\\27_11_22_w5\\2022-11-27_16-06_w5_Ch0\\output\\skeleton_spline_K_signed.csv',
 'Y:\\jalaja\\behaviour\\Autoscope\\data\\27_11_22_w6\\2022-11-27_17-01_w6_Ch0\\output\\skeleton_spline_K_signed.csv',
 'Y:\\jalaja\\behaviour\\Autoscope\\data\\27_11_22_w7\\2022-11-27_17-50_w7_Ch0\\output\\skeleton_spline_K_signed.csv',
 'Y:\\jalaja\\behaviour\\Autoscope\\data\\29_1_23_w1\\2023-01-29_13-07_W1_Ch0\\output\\skeleton_spline_K_signed.csv',
 'Y:\\jalaja\\behaviour\\Autoscope\\data\\29_1_23

In [9]:
def resample_with_interp(original_vector, new_length):
    # Find non-NaN indices
    non_nan_indices = ~np.isnan(original_vector)
    # Create a new set of indices for interpolation
    new_indices = np.linspace(0, len(original_vector) - 1, new_length)
    # Perform linear interpolation, handling NaN values
    resampled_vector = np.interp(new_indices, np.where(non_nan_indices)[0], original_vector[non_nan_indices])
    return resampled_vector

def plot_kymogram(kymo_path):    
    #kymo_path = os.path.join(project_path, 'skeleton_spline_K.csv')
    df_kymo = pd.read_csv(kymo_path, header=None)
    df_kymo_filled= df_kymo.fillna(method='ffill', limit=10) 
    # Forward fill any remaining NaN values at the start (if needed)
    head_osillation=detect_oscillations(df_kymo_filled)
        # print(df_kymo.shape)
    fig, axes = plt.subplots(dpi=150, figsize=(8, 6))  # dpi=400, figsize=(40,4),)
        # fig.suptitle(kymo_path)
    axes.imshow(df_kymo.T.loc[:,14000:14400], origin="upper", cmap='Spectral', aspect='auto', vmin=-0.01, vmax=0.01)

In [10]:
results_df = pd.DataFrame()
dor_vent_df=pd.DataFrame()
for file in spline_files:
    # Read the CSV file
    df_kymo = pd.read_csv(file, delimiter=',')
    # Interpolate NaN values from both ends
    # Fill NaNs by first forward-filling, then backward-filling along columns
    df_kymo_filled= df_kymo.fillna(method='ffill', limit=10)  # Only fill the first two NaNs in a gap
    
    curvature = np.sum(df_kymo_filled.abs(), axis=1)
    #head is defined as first 10 segments
    signed_head_curvature=np.sum(df_kymo_filled.iloc[:,1:10], axis=1)

    if df_kymo.shape[0] < 18950:
        total_curvature = np.rad2deg(curvature)
        head_curvature=signed_head_curvature
    else:
        curvature = resample_with_interp(curvature, 18900)
        total_curvature = np.rad2deg(curvature)
        head_curvature=resample_with_interp(signed_head_curvature, 18900)

    curvature_df = pd.DataFrame(total_curvature) # Convert to DataFrame
    head_curvature_df=pd.DataFrame(head_curvature)
    results_df = pd.concat([results_df, curvature_df], axis=1)
    dor_vent_df= pd.concat([dor_vent_df, head_curvature_df], axis=1)
    # plt.plot(total_curvature)
    # plot_kymogram(file)
    # plt.show()

In [11]:
results_df

,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0,20.246127,25.732083,22.885236,18.468474,25.744319,23.361030,28.868792,33.809911,25.306026,30.733125,...,29.163575,41.978144,26.696571,24.019547,32.216167,23.106340,26.384565,34.822245,28.378499,23.700052
1,21.717368,25.403490,23.706743,18.682847,28.219134,22.714011,28.777879,35.300807,23.976819,35.421535,...,29.738334,43.297059,27.014207,24.704911,31.641221,27.163186,24.935679,31.698313,30.442631,21.576093
2,20.696712,25.575134,25.733406,19.806698,24.738527,22.020560,29.568129,38.385885,25.441716,31.938336,...,31.030634,41.243350,27.023945,26.935939,31.312941,26.589127,27.465546,34.578460,31.084471,21.521625
3,16.660999,25.474328,28.915578,18.420051,24.943614,19.865485,28.755010,34.894591,23.715985,29.337238,...,30.255391,36.086859,25.780003,26.628293,35.218479,27.499053,24.888123,34.335865,35.118715,22.032401
4,16.259001,26.980046,34.510465,19.186647,23.906097,20.004895,30.271035,46.204525,23.815691,29.120771,...,28.944886,42.186311,26.110558,29.634096,33.998530,23.880530,30.088201,32.044369,35.562513,23.712695
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18896,24.585520,22.322464,25.151399,33.403306,28.509126,19.585819,28.082535,24.558380,NaN,27.436349,...,26.368935,34.944464,31.521298,26.811805,33.233569,20.941504,46.565925,54.282255,22.860538,25.990944
18897,25.968368,22.427251,23.514268,27.168086,34.434871,21.928379,24.352185,25.420289,NaN,28.303238,...,28.877303,34.316219,35.207173,29.093493,30.011589,19.171201,47.050987,81.820237,23.124803,26.467432
18898,23.866753,22.213060,23.701714,27.361168,33.570753,24.953153,24.388726,26.352829,NaN,27.932201,...,24.550529,33.363046,31.689955,23.146278,32.656554,21.237670,60.168582,81.820237,22.979089,24.671273
18899,25.975360,24.785458,22.695080,27.392125,31.292241,23.883894,24.128828,33.629314,NaN,NaN,...,26.134486,41.510854,31.024928,24.916823,31.836500,23.199242,57.088718,81.820237,24.300815,26.028520


In [13]:
results_df.to_csv(r"C:\Users\Jalaja Madhusudhanan\Desktop\autoscope_stage position\total_curvature.csv",header=None,index=None)

In [27]:
dor_vent_df.to_csv(r"C:\Users\Jalaja Madhusudhanan\Desktop\autoscope_stage position\head_curvature.csv",header=None,index=None)

In [7]:
def detect_oscillations(df_kymo):
    """
    Detects if there is at least one sign switch (negative to positive or vice versa) 
    within the first 20 columns of each row in df_kymo. If there is a sign switch, 
    it marks 1, otherwise 0. If any NaN is found, it returns NaN for that row.
    
    Parameters:
    df_kymo (pd.DataFrame): The input dataframe containing head oscillation data.

    Returns:
    pd.Series: A Series with 0 (no sign switch), 1 (sign switch detected), and NaN.
    """
    def check_sign_switch(row):
        # Slice the first 20 columns
        row = row[1:10]

        # If there's any NaN in the row, return NaN
        if row.isna().any():
            return np.nan
        
        # Check for sign changes
        sign_change = np.sign(row).diff().abs().sum() > 0
        
        return 1 if sign_change else 0

    # Apply the function row-wise and return a Series
    head_oscillation=df_kymo.T.apply(check_sign_switch)
    # Create a binary vector where the onset occurs (where head_oscillation transitions from 0 to 1)
    oscillation_onsets = np.diff(head_oscillation, prepend=0)
    
    # Convert to a binary vector: 1 for transition from 0 to 1, otherwise 0
    oscillation_onsets = np.where(oscillation_onsets > 0, 1, 0)
    return head_oscillation, oscillation_onsets


In [8]:
df_kymo.T[2:15]

NameError: name 'df_kymo' is not defined

In [ ]:
def plot_head_oscillation(df_kymo, df_osci):
    # Set up the figure with two subplots
    fig, (ax1, ax2) = plt.subplots(nrows=2, dpi=150, figsize=(8, 6), gridspec_kw={'height_ratios': [1, 3]})
    
    plt.subplots_adjust(hspace=0)  
    # Reshape the oscillation data for plotting
    df_osci = df_osci.values.reshape(1, -1)
    df_osci_sliced = df_osci[:, 14000:14700]
    # Plot the head oscillation (top)
    im=ax1.imshow(df_osci_sliced, origin="upper", cmap='binary', 
               vmin=0, vmax=1, aspect='auto')
    cbar = fig.colorbar(im, ax=ax1, orientation='vertical')
    ax1.set_axis_off()
    
    # Plot the kymograph (bottom)
    ax2.imshow(df_kymo.T.loc[:,14000:14700], origin="upper", cmap='Spectral', extent=[0, df_kymo.shape[0]/5, df_kymo.shape[1], 0], aspect=10, vmin=-0.01, vmax=0.01)
    ax2.set_axis_off()
    
    # Return the figure object for further use
    return fig

# Example usage
# fig = plot_head_oscillation(df_kymo, head_oscillation)
# plt.show()


In [ ]:
oscillation_onsets

In [ ]:
# just a better way to plot 
import numpy as np
import matplotlib.pyplot as plt

def plot_head_oscillation(df_kymo, df_osci, start_col=14000, end_col=14400):
    # Set up a single plot
    fig, ax = plt.subplots(dpi=150, figsize=(8, 6))
    
    # Slice the kymograph and oscillation data to the same range
    df_kymo_sliced = df_kymo.T.iloc[:, start_col:end_col]
    df_osci_sliced = df_osci.iloc[start_col:end_col]

    # Detect head oscillation onsets: where df_osci transitions from 0 to 1
    oscillation_onsets = np.where(np.diff(df_osci_sliced) > 0)[0] + 1  # Add 1 to correct shift caused by diff

    # Plot the kymograph (background)
    im = ax.imshow(df_kymo_sliced, origin="upper", cmap='Spectral', 
                   aspect='auto', vmin=-0.01, vmax=0.01)
    

    # Plot the head oscillation onsets as vertical lines on the same x-axis
    for onset in oscillation_onsets:  # For each detected oscillation onset
        ax.axvline(x=onset, color='black', linewidth=1)  # Draw vertical lines at oscillation onsets

    # Set labels
    # ax.set_xlabel('Time (frames)')
    # ax.set_ylabel('# Body Segment')
    # ax.set_title("Kymograph with Head Oscillation Onsets")
    
    return fig
# Example usage:
# fig = plot_head_oscillation(df_kymo, df_osci, start_col=14000, end_col=14700)
# plt.show()


In [ ]:
kymo_path=(r'Z:\\jalaja\\behaviour\\Autoscope\\data\\27_11_22_w1\\2022-11-27_12-31_w1_Ch0\\output\\skeleton_spline_K_signed.csv')
df_kymo = pd.read_csv(kymo_path, header=None)
 # Detect head oscillations (sign switch detection)
head_oscillation, oscillation_onsets= detect_oscillations(df_kymo)


In [ ]:
plot_kymogram(kymo_path)
plot_head_oscillation(df_kymo,head_oscillation)

In [ ]:
results_df = pd.DataFrame()

def resample_binary_signal(signal, target_size):
    """Resample binary signal and keep it binary by thresholding."""
    resampled_signal = resample_with_interp(signal, target_size)
    return np.where(resampled_signal > 0.5, 1, 0)  # Ensure binary by thresholding

for file in spline_files:
    # Read the CSV file
    df_kymo = pd.read_csv(file, delimiter=',')
    
    # Detect head oscillations and onsets (returns binary values and indices of onsets)
    head_oscillations, oscillation_onsets = detect_oscillations(df_kymo)

    # If the number of frames is less than 18950, use it as is, else resample
    if df_kymo.shape[0] < 18950:
        oscillation_frequency = pd.Series(oscillation_onsets).rolling(window=10, min_periods=1).sum()
    else:
        oscillation_resampled = resample_binary_signal(oscillation_onsets, 18900)
        oscillation_frequency = pd.Series(oscillation_resampled).rolling(window=10, min_periods=1).sum()

    # Convert the oscillations into a DataFrame
    oscillation_df = pd.DataFrame(oscillation_frequency)
    
    # Concatenate the oscillation results
    results_df = pd.concat([results_df, oscillation_df], axis=1)

# Now 'results_df' contains oscillation frequencies for each file.
results_df

In [ ]:
results_df.to_csv(r"C:\Users\Jalaja Madhusudhanan\Desktop\autoscope_stage position\head_oscillation_frequency.csv",header=None,index=None)

In [ ]:
# tried to use Harris's published method to find head oscillation/head cast
# didn't work
from scipy.signal import find_peaks

def find_local_extrema_per_column(df_kymo, min_prominence=0.01, min_distance=50):
    """
    Iteratively find local maxima and minima for each column (segment) in the df_kymo dataframe.
    Save the results in a DataFrame and plot them as an event plot.
    
    :param df_kymo: DataFrame representing kymograph data, where each column is a segment.
    :param min_prominence: Minimum prominence of the peaks.
    :param min_distance: Minimum distance between peaks.
    :return: DataFrame with 1 marking maxima/minima and 0 otherwise.
    """
    # Initialize a DataFrame to store results (same shape as df_kymo, filled with 0s)
    extrema_df = pd.DataFrame(0, index=df_kymo.index, columns=df_kymo.columns)
    
    # Iterate over each column (segment) in the DataFrame
    for col in df_kymo.columns:
        segment = df_kymo[col].to_numpy()  # Convert column to numpy array
        
        # Find local maxima and minima for this segment
        maxima_indices, _ = find_peaks(segment, prominence=min_prominence, distance=min_distance)
        minima_indices, _ = find_peaks(-segment, prominence=min_prominence, distance=min_distance)
        
        # Mark maxima and minima as 1 in the DataFrame
        extrema_df.loc[maxima_indices, col] = 1  # Mark maxima
        extrema_df.loc[minima_indices, col] = 1  # Mark minima
    
    return extrema_df

def plot_extrema_eventplot(extrema_df):
    """
    Plot the local maxima and minima as an event plot.
    
    :param extrema_df: DataFrame with 1 marking maxima/minima and 0 otherwise.
    """
    fig, ax = plt.subplots(dpi=150, figsize=(10, 6))
    
    # Prepare the event plot data
    event_data = []
    for col in extrema_df.columns:
        event_data.append(extrema_df.index[extrema_df[col] == 1].to_numpy())
    
    # Plot the event data (one row for each segment/column)
    ax.eventplot(event_data, orientation='horizontal', colors='black', linelengths=0.9)
    ax.set_title('Local Maxima and Minima Event Plot')
    ax.set_xlabel('Frame')
    ax.set_ylabel('Segments')
    plt.show()

# Example usage
extrema_df = find_local_extrema_per_column(df_kymo, min_prominence=0.005, min_distance=10)
plot_extrema_eventplot(extrema_df)

# Optionally, you can inspect the result DataFrame
print(extrema_df.head())


In [15]:
# dorsal-ventral turn quantification
manual_anno=pd.read_csv(r"C:\Users\Jalaja Madhusudhanan\Desktop\autoscope_stage position\manual_annotation.csv",header=None)

In [13]:
def find_turn_onset(df):
    # Create a copy of the DataFrame to store the results
    result_df = pd.DataFrame(0, index=df.index, columns=df.columns)
    
    for col in df.columns:
        data = df[col].values
        switch_indices = np.where((data[:-1] != 2) & (data[1:] == 2))[0] + 1  # Find indices where switch to 2 occurs
        
        result_df.loc[switch_indices, col] = 1  # Mark those indices with 1 in the result DataFrame
    
    return result_df

In [16]:
turns =find_turn_onset(manual_anno)

In [20]:
turns.sum()

0     23
1     48
2     44
3     34
4     47
5     53
6     19
7     39
8     16
9     13
10    10
11    31
12    20
13    19
14    27
15    23
16    47
17    34
18    39
19    30
20    18
21    55
22    28
23    43
24    23
dtype: int64

In [18]:
# Create a new DataFrame to store the labels
labels_df = pd.DataFrame(index=dor_vent_df.index, columns=dor_vent_df.columns)

# Iterate over each column in the DataFrames
for col in dor_vent_df.columns:
    # Get the indices where the value is 1 in results_df
    switch_indices = turns[col] == 1
    
    # Align indices before fetching values
    switch_indices = switch_indices.reindex(dor_vent_df.index, fill_value=False)
    
   # Fetch corresponding values from dor_vent_df at these indices
    values_at_switch = dor_vent_df.loc[switch_indices, col]

   # Use np.where to label as 'dorsal' if negative, 'ventral' if positive, and keep NaN if the value is NaN
    labels = np.where(values_at_switch < 0, 'dorsal',
                  np.where(values_at_switch > 0, 'ventral', np.nan))


    # Store the labels in the new DataFrame
    labels_df.loc[switch_indices, col] = labels

# The labels_df now contains 'dorsal', 'ventral', or NaN based on the value in dor_vent_df


In [19]:
labels_df

,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18896,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18897,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18898,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18899,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
